In [1]:
from dataclasses import dataclass
from datetime import datetime
from typing import Optional, Dict, Any, List

In [ ]:
class PetProfile:
    """
    Pet data
    """
    pet_id: str
    breed: str            # for searching any special attribute of the breed
    birth_date: datetime
    is_high_energy: bool  # e.g., Vizsla 
    postcode: str         # For seasonal/geographic tick risk 
    streak: int           # auto calculate number of week checkin in (consecutive), reset if hasn't checkin more than 1 week
    consistency: bool     # Automatic check if has owner has checkin regularly

@dataclass
class CheckInData:
    """Pet check-in data, recorded by owner (moment 1)"""
    pet_id: str # record automatically after create pet profile
    timestamp: datetime    # Record check-in time
    appetite: int          # 1-10 scale
    energy: int            # 1-10 scale
    sleep_quality: int     # 1-10 scale
    water_intake: str      # "Normal", "Increased" 
    toileting_blood: bool  # Acute override trigger 
    activity_minutes: int  # Still figuring how to actually track this 
    behaviour_and_mood: str
    physical_health: [eyes: int,  coat: int, ears: int] # scale 1-10 for each
    owner_struggling: bool # N/A for score, but flags dashboard that might need for Evie to jump in and start giving advice for quick action

# Tick season
class TickSeason:
    """Tick season data"""
    tick_season: bool
# Vet 

# Vaccination

# Check_in consistency

# Vet visit

    

In [ ]:
def pain_indicators():
    """Combined signal by moment 1 or trigger by pet owner"""
    return False

In [ ]:
class VetInteractionLog:
    """List of yes and no question and will be input in this to account to Vitality score"""

In [ ]:
class PetDiseaseIndicator:
    """
    Evaluates check-in data specifically for clinical red flags and maps them to potential disease categories for vet reference.
    """
    def __init__(self, pet: PetProfile):
        self.pet = pet

    def evaluate(self, current: CheckInData) -> List[Dict[str, str]]:
        clinical_flags = []

        # Increased Water Intake
        if current.water_intake == "Increased":
            clinical_flags.append({
                "trigger": "Increased Water Intake",
                "severity": "High",
                "action": "Requires bloodwork and urinalysis."
            })

        # Blood in Stool
        if current.toileting_blood:
            clinical_flags.append({
                "trigger": "Blood in Stool",
                "severity": "Acute",
                "action": "Immediate veterinary attention recommended."
            })

        # Confirmed Tick
        if current.tick_found:
            clinical_flags.append({
                "trigger": "Tick Found",
                "severity": "Acute",
                "action": "Immediate search for additional ticks and symptom monitoring; vet visit if symptomatic."
            })

        # Sudden Energy/Appetite Collapse 
        if current.energy <= 1 and current.appetite <= 1:
            clinical_flags.append({
                "trigger": "Lethargy & Anorexia Cluster",
                "severity": "Acute",
                "action": "Immediate veterinary attention recommended."
            })

        return clinical_flags

For now weight are indicated manually, weighting may changes base on pet age, breed,...

In [ ]:
class VitalityScoreEngine:
    """
    Vitality score calculator, using 4 week scores, need veterinians review for weight adjustment (this is just a demo)
    """
    def __init__(self, pet: PetProfile, history: list):
        self.pet = pet
        self.history = history # Previous 4 weeks for baseline (should use only 4 previous week
    def calculate_score(self, current: 'CheckInData') -> Dict[str, Any]:
        # 1. ACUTE OVERRIDES (Bypasses incremental scoring) 
        # "sudden collapse in appetite combined with lethargy" OR blood OR increased water
        is_lethargic_collapse = (current.energy <= 1 and current.appetite <= 1)
        if current.toileting_blood or current.water_intake == "Increased" or is_lethargic_collapse:
            return {"score": 40, "zone": "Action Needed", "override": True}

        # 2. BASELINE GATEKEEPER 
        # The first four weeks are a baseline establishment period.
        if len(self.history) < 4:
            return {"score": None, "status": "Building your baseline"}

        # 3. ESTABLISH THE PET'S "NORMAL" (Comparative averages)
        avg_energy_score = sum(h.energy for h in self.history) / 4
        avg_appetite_score = sum(h.appetite for h in self.history) / 4
        avg_sleep_score = sum(h.sleep_quality for h in self.history) / 4
        avg_activity_score = sum(h.activity_minutes for h in self.history) / 4

        # Calculate negative deviations (drops from normal)
        drop_energy = max(0, avg_energy_score - current.energy)
        drop_appetite = max(0, avg_appetite_score - current.appetite)
        drop_sleep = max(0, avg_sleep_score - current.sleep_quality)
        
        # Calculate activity drop as a percentage ratio
        drop_act_ratio = max(0, (avg_activity_score - current.activity_minutes) / max(1, avg_activity_score))

        # 4. ASSIGN BASE WEIGHTINGS (Per Summary Table)
        # Default: Medium-High (0.25) across the board.
        w_energy = 0.25
        w_appetite = 0.25
        w_sleep = 0.25
        w_activity = 0.25
        
        # High-energy breeds carry elevated activity weighting.
        if self.pet.is_high_energy:
            w_activity = 0.40 
            w_sleep = 0.10 # Adjust sleep down to balance the scale

        raw_score = 100.0

        # 5. DETECT SIGNALS VS NOISE (Elevated Combination Penalties)
        # Elevated when combined with other Moment 1 signals.
        has_appetite_signal = drop_appetite >= 1.0
        has_energy_signal = drop_energy >= 1.0
        has_sleep_signal = drop_sleep >= 1.0

        if has_appetite_signal and (has_energy_signal or has_sleep_signal):
            # This is a confirmed signal, apply a flat heavy deduction
            raw_score -= 25.0
        else:
            # This is standard variance/noise, apply normal weighted deductions
            raw_score -= (drop_appetite * 10 * w_appetite)
            raw_score -= (drop_energy * 10 * w_energy)
            raw_score -= (drop_sleep * 10 * w_sleep)

        # Always deduct for major activity drops based on breed weighting
        raw_score -= (drop_act_ratio * 30 * w_activity)

        # 6. FINALIZE SCORE
        final_score = max(0, min(100, int(raw_score)))

        return {
            "score": final_score, 
            "zone": self._get_zone(final_score),
            "status": "Active"
        }

    def _get_zone(self, score: int) -> str:
        """Returns the colour-coded zone to trigger Evie"""
        if score >= 85: return "Bright Green"
        if score >= 70: return "Medium Green"
        if score >= 50: return "Yellow/Watch Zone"
        return "Action Needed"
        